#In this notebook, we compare performance of survival models to predict loan prepayment, default, or continuance.

We compare the Random Forest, LSTM, and TabNet performance. We measure success on test set PR-AUC and KS-score.

This notebook used Perplexity AI for error correction and basic code generation


The following cells involve data preprocessing.

In [ ]:
#Manipulating Prosper Loan Data to Recover Prepayment and Default Information
import pandas as pd

file_path = 'prosperLoanData.csv'
df = pd.read_csv(file_path,engine='python', on_bad_lines='skip')
df.head()

#Clean up time objects
df['ListingCreationDate'] = pd.to_datetime(df['ListingCreationDate'],  format='mixed').dt.date
df['ClosedDate'] = pd.to_datetime(df['ClosedDate'],  format='mixed').dt.date

#Eliminate loans that were cancelled, as these are not relevant for our analysis of prepayment and default risk.
df = df[~df['LoanStatus'].isin(['Cancelled'])]

#Create a column to determine if a loan was prepaid
df['CompletedLoans'] = df['LoanStatus'].isin(['Completed'])
df['Prepayment'] = df['CompletedLoans'] & ((pd.to_datetime(df['ClosedDate']) - pd.to_datetime(df['LoanOriginationDate'])).dt.days / 30 < df['Term']-1)
#Create a column to indicate if a loan was prepaid
df['IsPrepaid'] = df['Prepayment'].astype(int)

#If a loan was prepaid, change the loan status to 'Prepaid'
df.loc[df['IsPrepaid'] == 1, 'LoanStatus'] = 'Prepaid'

#Create a new column that codifies the loan status.
#Assign 0 for Current Loans, 1 for Completed Loans, 2 for Defaulted Loans, and 3 for Prepaid Loans.
def categorize_loan_status(status):
    if status in ['Current', 'Past Due (1-15 days)', 'Past Due (31-60 days)', 'Past Due (16-30 days)',  'Past Due (61-90 days)', 'Past Due (91-120 days)', 'Past Due (>120 days)', 'Past Due (181-360 days)', 'Past Due (16-30 days)', 'Past Due (31-60 days)', 'Past Due ( days)', 'Past Due (Over 720 days)', 'FinalPaymentInProgress']:
        return 0
    elif status in ['Completed', 'FinalPaymentInProgress']:
        return 1
    elif status in ['Defaulted', 'Chargedoff']:
        return 2
    elif status in ['Prepaid']:
        return 3
    else:
        return -1  # For any other statuses not covered above

df['LoanStatusCategory'] = df['LoanStatus'].apply(categorize_loan_status)
#Create a column that indicates how many months it took to repay or default a loan
df['MonthsToRepayOrDefault'] = ((pd.to_datetime(df['ClosedDate']) - pd.to_datetime(df['LoanOriginationDate'])).dt.days / 30).fillna(-1)
#Round MonthsTORepayOrDefault to nearest integer
df['MonthsToRepayOrDefault'] = df['MonthsToRepayOrDefault'].round()


#Look at the loan that defaulted with a negative MonthsToRepayOrDefault
df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)]
#throw this single data point out as a missed input
df = df.drop(df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)].index)
#Check that the data point has been removed
df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)]


#Check that only current loans have a negative MonthsToRepayOrDefault
df[df['MonthsToRepayOrDefault'] < 0]['LoanStatus'].value_counts()

#Value count for each event
df['LoanStatusCategory'].value_counts()



,count
LoanStatusCategory,
0,58848
3,27484
2,17009
1,10590


In [ ]:
#We want to make sure that there is no data on months for current/prepaid/defaulted loans past their last observed month.
import pandas as pd
import numpy as np

as_of_date = pd.to_datetime("2014-03-10")
df["ListingCreationDate"] = pd.to_datetime(df["ListingCreationDate"])

df["MonthsFromOrigToAsOf"] = ((as_of_date.year - df["ListingCreationDate"].dt.year) * 12 +
                              (as_of_date.month - df["ListingCreationDate"].dt.month))

df["LastObservedMonth"] = np.where(
    df["LoanStatusCategory"].isin([2, 3]),
    df["MonthsToRepayOrDefault"],
    df["MonthsFromOrigToAsOf"]
)

for month in range(1, 37):
    col_name = f"Month_{month}"
    event_indicator = np.zeros(len(df), dtype=float)
    prepay_mask = (df["LoanStatusCategory"] == 3) & (df["MonthsToRepayOrDefault"] == month)
    default_mask = (df["LoanStatusCategory"] == 2) & (df["MonthsToRepayOrDefault"] == month)

    event_indicator[prepay_mask] = 1
    event_indicator[default_mask] = 2

    # Now NaN assignment works
    censored_mask = df["LastObservedMonth"] < month
    event_indicator[censored_mask] = np.nan

    df[col_name] = event_indicator

df.head()



,ListingKey,ListingNumber,ListingCreationDate,CreditGrade,Term,LoanStatus,ClosedDate,BorrowerAPR,BorrowerRate,LenderYield,...,Month_27,Month_28,Month_29,Month_30,Month_31,Month_32,Month_33,Month_34,Month_35,Month_36
0,1021339766868145413AB3B,193129,2007-08-26,C,36,Prepaid,2009-08-14,0.16516,0.1580,0.1380,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10273602499503308B223C1,1209647,2014-02-27,NaN,36,Current,NaT,0.12016,0.0920,0.0820,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0EE9337825851032864889A,81716,2007-01-05,HR,36,Completed,2009-12-17,0.28269,0.2750,0.2400,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0EF5356002482715299901A,658116,2012-10-22,NaN,36,Current,NaT,0.12528,0.0974,0.0874,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0F023589499656230C5E3E2,909464,2013-09-14,NaN,36,Current,NaT,0.24614,0.2085,0.1985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
fed_df = pd.read_csv("FEDFUNDS (1).csv")
fed_df["observation_date"] = pd.to_datetime(fed_df["observation_date"])
fed_df = fed_df.sort_values("observation_date").drop_duplicates("observation_date").reset_index(drop=True)

print("FED data preview:")
print(fed_df.head())

# 3-class: 0=current/completed, 1=default, 2=prepay
df["Event36"] = np.where(
    df["LoanStatusCategory"].isin([0, 1]),  # current or completed
    0,
    np.where(
        df["LoanStatusCategory"] == 2,      # defaulted/chargedoff
        1,
        2                                   # prepaid
    )
)


FED data preview:
  observation_date  FEDFUNDS
0       1954-07-01      0.80
1       1954-08-01      1.22
2       1954-09-01      1.07
3       1954-10-01      0.85
4       1954-11-01      0.83


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score
import joblib

# ===== 1. FED PREP =====
fed_df = pd.read_csv("FEDFUNDS (1).csv")
fed_df["observation_date"] = pd.to_datetime(fed_df["observation_date"])
#Calculate fed rate at loan origin for each loan.
fed_monthly = fed_df.set_index('observation_date').resample('MS').interpolate('linear').ffill()
fed_dict = fed_monthly['FEDFUNDS'].to_dict()
print(f"FED ready: {len(fed_dict)} months")

# ===== 2. MONTHLY DATAFRAME =====
#Add month-by-month data for each loan, including the borrower rate, FED rate, spread/incentive info, and previous 3 month FED change.

monthly_rows = []
zero_apr_loans = 0

for idx, loan in df.iterrows():
    borrower_apr = loan['BorrowerRate']
    #Throw out "weird" loans with zero or negative APR
    if borrower_apr <= 0:
        zero_apr_loans += 1
        continue

    #Only add info on last observed month
    last_month = int(loan['LastObservedMonth'])
    #Gather data on each month since loan origination
    for month in range(1, min(37, last_month + 1)):
        month_date = loan['ListingCreationDate'] + pd.DateOffset(months=month-1)
        #retrieve fed rate for this month
        fed_now = fed_dict.get(month_date, 0.02)

        # Look at the FED rate 3 months prior (capturing some volatility)
        fed_3mo_ago_date = month_date - pd.DateOffset(months=3)
        fed_3mo_ago = fed_dict.get(fed_3mo_ago_date, fed_now)

        #Calculate the borrower's incentive to refinance this month
        apr_fed_spread = borrower_apr - fed_now
        #Compare with 3 months ago
        apr_fed_spread_change = (borrower_apr - fed_3mo_ago) - apr_fed_spread
        #Calculate fed change
        fed_change_3m = fed_now - fed_3mo_ago

        #Add all info into month-by-month dataframe.
        monthly_rows.append({
            'loan_id': idx,
            'cohort_year': loan['ListingCreationDate'].year,
            'month_since_orig': month,
            'status_next': loan[f'Month_{month}'],
            'apr_fed_spread': apr_fed_spread,
            'apr_fed_spread_change_3m': apr_fed_spread_change,
            'apr_fed_spread_pct': apr_fed_spread / borrower_apr,
            'fed_change_3m': fed_change_3m,
            'fed_level': fed_now,
            'borrower_apr': borrower_apr,
            'loan_amount_log': np.log1p(loan.get('LoanOriginalAmount', 10000))
        })

monthly_df = pd.DataFrame(monthly_rows)
print(f"Dataset: {len(monthly_df):,} loan-months ({zero_apr_loans} zero-APR skipped)")
print("Events:", dict(monthly_df['status_next'].value_counts()))


FED ready: 787 months
Dataset: 1,662,492 loan-months (8 zero-APR skipped)
Events: {0.0: np.int64(1619101), 1.0: np.int64(27065), 2.0: np.int64(16326)}


Following data preprocessing and organization, we prepare train/validation/test sets.

In [ ]:
selected_features = ['month_since_orig', 'apr_fed_spread', 'fed_level', 'borrower_apr', 'loan_amount_log']

In [ ]:
# Combined data: predict BOTH default AND prepay hazards
# Temporal split: train/valid/test by cohort_year
#Smaller train set to prevent overfit in the DL models

train_mask = monthly_df['cohort_year'] <= 2010
valid_mask = monthly_df['cohort_year'] == 2011
test_mask  = monthly_df['cohort_year'] >= 2012

# Multi-task targets
y_def_train = (monthly_df[train_mask]['status_next'] == 1).astype(int)
y_prepay_train = (monthly_df[train_mask]['status_next'] == 2).astype(int)

y_def_valid = (monthly_df[valid_mask]['status_next'] == 1).astype(int)
y_prepay_valid = (monthly_df[valid_mask]['status_next'] == 2).astype(int)

y_def_test = (monthly_df[test_mask]['status_next'] == 1).astype(int)
y_prepay_test = (monthly_df[test_mask]['status_next'] == 2).astype(int)

X_train = monthly_df[train_mask][selected_features].values
X_valid = monthly_df[valid_mask][selected_features].values
X_test  = monthly_df[test_mask][selected_features].values


# **Train Random Forest on 5 Features**



In [ ]:
#Drop the non-observed months
risk_df = monthly_df[monthly_df['status_next'].notna()]
prepay_data = risk_df[risk_df['status_next'].isin([0,2])]

#Train Random Forests
# Train finals on train data only (2005-2010)
train_mask = risk_df['cohort_year'] <= 2010

def_data_final = risk_df[(risk_df['status_next'].isin([0, 1])) & train_mask]
prepay_data_final = risk_df[(risk_df['status_next'].isin([0, 2])) & train_mask]

print(f"Final train cohorts - Default: {sorted(def_data_final['cohort_year'].unique())}")
print(f"Final train cohorts - Prepay:  {sorted(prepay_data_final['cohort_year'].unique())}")

# Default model
rf_default_final = RandomForestClassifier(
    n_estimators=500,
    class_weight={0: 1, 1: 25},
    max_depth=8,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

rf_default_final.fit(
    def_data_final[selected_features],
    (def_data_final['status_next'] == 1).astype(int),
)

# Prepay model
rf_prepay_final = RandomForestClassifier(
    n_estimators=500,
    class_weight={0: 1, 1: 6},
    max_depth=8,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

rf_prepay_final.fit(
    prepay_data_final[selected_features],
    (prepay_data_final['status_next'] == 2).astype(int),
)

Final train cohorts - Default: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010)]
Final train cohorts - Prepay:  [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010)]


RandomForestClassifier(class_weight={0: 1, 1: 6}, max_depth=8,
                       min_samples_leaf=100, n_estimators=500, n_jobs=-1,
                       oob_score=True, random_state=42)

In [ ]:
# 1) Default and prepay probabilities from RFs
risk_df = monthly_df[monthly_df['status_next'].notna()].copy()

risk_df['rf_default_prob'] = rf_default_final.predict_proba(
    risk_df[selected_features]
)[:, 1]

risk_df['rf_prepay_prob'] = rf_prepay_final.predict_proba(
    risk_df[selected_features]
)[:, 1]


train_mask = risk_df['cohort_year'] <= 2010
valid_mask = risk_df['cohort_year'] == 2011
test_mask  = risk_df['cohort_year'] >= 2012

# targets for tabnet
y_def_train = (risk_df[train_mask]['status_next'] == 1).astype(int)
y_def_valid = (risk_df[valid_mask]['status_next'] == 1).astype(int)
y_def_test  = (risk_df[test_mask]['status_next'] == 1).astype(int)


In [ ]:
#Test RF Performance
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

def evaluate_rf_lift(rf_model, X_test, y_def_test, y_prepay_test):
    """Exact equivalent of your TabNet comprehensive test for RF"""

    RF_train = monthly_df[train_mask][selected_features].values
    RF_valid = monthly_df[valid_mask][selected_features].values
    RF_test = monthly_df[test_mask][selected_features].values
    # RF predictions
    default_pred = rf_model.predict_proba(RF_test)[:, 1]

    # DEFAULT metrics
    def_roc = roc_auc_score(y_def_test, default_pred)
    def_pr = average_precision_score(y_def_test, default_pred)
    baseline_def = y_def_test.mean()

    # PREPAY metrics
    prepay_roc = roc_auc_score(y_prepay_test, default_pred)
    prepay_pr = average_precision_score(y_prepay_test, default_pred)
    baseline_prepay = y_prepay_test.mean()

    print("=== RF DEFAULT + PREPAY PERFORMANCE ===")
    print(f"DEFAULT (status_next=1):")
    print(f"  ROC-AUC: {def_roc:.3f}, PR-AUC: {def_pr:.3f}")
    print(f"  Baseline: {baseline_def:.4f}, PR Lift: {def_pr/baseline_def:.1f}x")

    print(f"\nPREPAY (status_next=2):")
    print(f"  ROC-AUC: {prepay_roc:.3f}, PR-AUC: {prepay_pr:.3f}")
    print(f"  Baseline: {baseline_prepay:.4f}, PR Lift: {prepay_pr/baseline_prepay:.1f}x")

    # TOP 10% LIFT
    top_decile = default_pred >= np.percentile(default_pred, 90)
    top_def_rate = y_def_test[top_decile].mean()
    top_prepay_rate = y_prepay_test[top_decile].mean()

    print(f"\n=== TOP 10% HIGHEST RF SCORES ===")
    print(f"Default:  {top_def_rate:.4f} ({top_def_rate/baseline_def:.1f}x)")
    print(f"Prepay:   {top_prepay_rate:.4f} ({top_prepay_rate/baseline_prepay:.1f}x)")
    print(f"Exit:     {(top_def_rate + top_prepay_rate):.4f}")

    return {
        'def_pr_lift': def_pr / baseline_def,
        'prepay_pr_lift': prepay_pr / baseline_prepay,
        'top10_def_lift': top_def_rate / baseline_def,
        'top10_prepay_lift': top_prepay_rate / baseline_prepay
    }

# === RUN FOR RF MODELS ===
# 2012-14 test sets
results_default = evaluate_rf_lift(
    rf_default_final, X_test, y_def_test, y_prepay_test
)

print("\n" + "="*50)
results_prepay = evaluate_rf_lift(  # Separate prepay model
    rf_prepay_final, X_test, y_def_test, y_prepay_test
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


=== RF DEFAULT + PREPAY PERFORMANCE ===
DEFAULT (status_next=1):
  ROC-AUC: 0.555, PR-AUC: 0.015
  Baseline: 0.0122, PR Lift: 1.2x

PREPAY (status_next=2):
  ROC-AUC: 0.617, PR-AUC: 0.006
  Baseline: 0.0050, PR Lift: 1.2x

=== TOP 10% HIGHEST RF SCORES ===
Default:  0.0163 (1.3x)
Prepay:   0.0043 (0.9x)
Exit:     0.0206



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


=== RF DEFAULT + PREPAY PERFORMANCE ===
DEFAULT (status_next=1):
  ROC-AUC: 0.553, PR-AUC: 0.014
  Baseline: 0.0122, PR Lift: 1.1x

PREPAY (status_next=2):
  ROC-AUC: 0.794, PR-AUC: 0.013
  Baseline: 0.0050, PR Lift: 2.5x

=== TOP 10% HIGHEST RF SCORES ===
Default:  0.0135 (1.1x)
Prepay:   0.0121 (2.4x)
Exit:     0.0256


In [ ]:
# RF KS
from sklearn.metrics import roc_curve
import numpy as np

def ks_stat(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return max(np.abs(tpr - fpr))

print("RF Prepay Test KS:", round(ks_stat(y_prepay_test, risk_df.loc[test_mask, 'rf_prepay_prob']), 3))
print("RF Default Test KS:", round(ks_stat(y_def_test, risk_df.loc[test_mask, 'rf_default_prob']), 3))


RF Prepay Test KS: 0.516
RF Default Test KS: 0.084


In [ ]:
tabnet_features = selected_features + ['rf_default_prob', 'rf_prepay_prob']

X_train = risk_df[train_mask][tabnet_features].values
X_valid = risk_df[valid_mask][tabnet_features].values
X_test  = risk_df[test_mask][tabnet_features].values

# **Train TabNet with 10 Features**

In [ ]:
#==== TabNet with RF features ====
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

# Scale numeric features (including RF probs)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)      # <=2010
X_valid_s = scaler.transform(X_valid)          # 2011
X_test_s  = scaler.transform(X_test)           # 2012-2014

# Class weights for severe imbalance
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_def_train),
    y=y_def_train,
)

multi_tabnet = TabNetClassifier(
    n_d=24, n_a=24, n_steps=5,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.01),
    gamma=1.3,
    verbose=1,
    seed=42,
)

multi_tabnet.fit(
    X_train_s, y_def_train,
    eval_set=[(X_valid_s, y_def_valid)],   # 2012-2014 validation
    eval_metric=['auc'],
    max_epochs=100,
    patience=15,
    weights={0: class_weights[0], 1: class_weights[1]},
)

ModuleNotFoundError: No module named 'pytorch_tabnet'

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import numpy as np

# FINAL 2014 TEST EVALUATION (after training complete)
def evaluate_final_test(model, scaler, X_test, y_def_test):
    """Complete unbiased evaluation on 2014 cohort"""

    # Predict on test set
    X_test_s = scaler.transform(X_test)
    y_pred_proba = model.predict_proba(X_test_s)[:, 1]

    # Core metrics
    test_roc_auc = roc_auc_score(y_def_test, y_pred_proba)
    test_pr_auc = average_precision_score(y_def_test, y_pred_proba)

    # Decile lift analysis (critical for simulation)
    decile_edges = np.percentile(y_pred_proba, np.arange(10, 100, 10))
    default_rate = y_def_test.mean()

    print("=== 2012 - 014 TEST RESULTS ===")
    print(f"ROC-AUC:     {test_roc_auc:.3f}")
    print(f"PR-AUC:      {test_pr_auc:.3f}")
    print(f"Baseline:    {default_rate:.4f}")
    print(f"PR Lift:     {test_pr_auc/default_rate:.1f}x")

    # Top decile performance
    top_decile_mask = y_pred_proba >= np.percentile(y_pred_proba, 90)
    top_decile_default_rate = y_def_test[top_decile_mask].mean()
    print(f"\nTop 10% Loans:")
    print(f"  Predicted:  {top_decile_mask.mean():.1%} of portfolio")
    print(f"  Default Rate: {top_decile_default_rate:.4f}")
    print(f"  Lift:       {top_decile_default_rate/default_rate:.1f}x")

    return {
        'roc_auc': test_roc_auc,
        'pr_auc': test_pr_auc,
        'top_decile_lift': top_decile_default_rate/default_rate
    }


results = evaluate_final_test(multi_tabnet, scaler, X_test, y_def_test)


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# COMPREHENSIVE TEST: Default + Prepay Performance
def test_model_comprehensive(model, scaler, X_test, y_def_test, y_prepay_test):
    """Test both default AND prepay prediction quality on 2014 test set"""

    # Scale test data
    X_test_s = scaler.transform(X_test)

    # Get predictions (TabNet default head predicts default probability)
    default_pred = model.predict_proba(X_test_s)[:, 1]

    # Test DEFAULT performance
    def_roc = roc_auc_score(y_def_test, default_pred)
    def_pr = average_precision_score(y_def_test, default_pred)

    # For PREPAY: Use SAME predictions (multi-task shared encoder)
    # Higher risk scores capture BOTH defaults + prepays
    prepay_roc = roc_auc_score(y_prepay_test, default_pred)
    prepay_pr = average_precision_score(y_prepay_test, default_pred)

    # Decile analysis
    baseline_def_rate = y_def_test.mean()
    baseline_prepay_rate = y_prepay_test.mean()

    print("=== 2012-2014 TEST SET PERFORMANCE ===")
    print(f"DEFAULT PREDICTION (status_next=1):")
    print(f"  ROC-AUC:     {def_roc:.3f}")
    print(f"  PR-AUC:      {def_pr:.3f}")
    print(f"  Baseline:    {baseline_def_rate:.4f}")
    print(f"  PR Lift:     {def_pr/baseline_def_rate:.1f}x")

    print(f"\nPREPAY PREDICTION (status_next=2):")
    print(f"  ROC-AUC:     {prepay_roc:.3f}")
    print(f"  PR-AUC:      {prepay_pr:.3f}")
    print(f"  Baseline:    {baseline_prepay_rate:.4f}")
    print(f"  PR Lift:     {prepay_pr/baseline_prepay_rate:.1f}x")

    # Top decile lift
    top_decile = default_pred >= np.percentile(default_pred, 90)

    top_def_rate = y_def_test[top_decile].mean()
    top_prepay_rate = y_prepay_test[top_decile].mean()

    print(f"\n=== TOP 10% RISKIEST LOANS ===")
    print(f"Default Rate:  {top_def_rate:.4f} ({top_def_rate/baseline_def_rate:.1f}x lift)")
    print(f"Prepay Rate:   {top_prepay_rate:.4f} ({top_prepay_rate/baseline_prepay_rate:.1f}x lift)")
    print(f"Exit Rate:     {(top_def_rate + top_prepay_rate):.4f}")

    return {
        'default': {'roc': def_roc, 'pr': def_pr},
        'prepay': {'roc': prepay_roc, 'pr': prepay_pr},
        'top_decile_lift_def': top_def_rate/baseline_def_rate,
        'top_decile_lift_prepay': top_prepay_rate/baseline_prepay_rate
    }

results = test_model_comprehensive(multi_tabnet, scaler, X_test, y_def_test, y_prepay_test)


# **Fitting LSTM Model on RF's Top 5 features**

In [ ]:
%%time
# === LSTM Preprocessing to Tensors ===
import numpy as np
from sklearn.preprocessing import LabelEncoder

MAX_SEQ_LEN = 36
selected_features = ['month_since_orig', 'apr_fed_spread', 'fed_level',
                    'borrower_apr', 'loan_amount_log']

print(f"risk_df: {risk_df.shape}")

# loan encoding
le = LabelEncoder()
risk_df_temp = risk_df.copy()
risk_df_temp['loan_idx'] = le.fit_transform(risk_df_temp['loan_id'])
month_idx = (risk_df_temp['month_since_orig'] - 1).clip(0, MAX_SEQ_LEN-1).values

num_loans = risk_df['loan_id'].nunique()
num_feats = len(selected_features)

# Initialize
X_seq = np.zeros((num_loans, MAX_SEQ_LEN, num_feats))
y_def_seq = np.zeros((num_loans, MAX_SEQ_LEN))
y_prepay_seq = np.zeros((num_loans, MAX_SEQ_LEN))
masks_seq = np.zeros((num_loans, MAX_SEQ_LEN))

# VECTORIZED FILL
loan_idx = risk_df_temp['loan_idx'].values
feat_vals = risk_df_temp[selected_features].fillna(0).values

for f in range(num_feats):
    X_seq[loan_idx, month_idx, f] = feat_vals[:, f]

y_def_seq[loan_idx, month_idx] = (risk_df_temp['status_next'] == 1).values
y_prepay_seq[loan_idx, month_idx] = (risk_df_temp['status_next'] == 2).values
masks_seq[loan_idx, month_idx] = 1.0

print(f"X {X_seq.shape}, events def:{y_def_seq.sum():.0f} prepay:{y_prepay_seq.sum():.0f}")
del risk_df_temp  # Memory


risk_df: (1662492, 13)
X (112627, 36, 5), events def:27065 prepay:16326
CPU times: user 344 ms, sys: 207 ms, total: 551 ms
Wall time: 545 ms


In [ ]:
%%time
cohort_map = risk_df.groupby('loan_id')['cohort_year'].first()
train_mask = cohort_map < 2010
valid_mask = cohort_map == 2011
test_mask = cohort_map >= 2012

X_train, masks_train = X_seq[train_mask], masks_seq[train_mask]
y_def_train, y_prepay_train = y_def_seq[train_mask], y_prepay_seq[train_mask]

X_valid, masks_valid = X_seq[valid_mask], masks_seq[valid_mask]
y_def_valid, y_prepay_valid = y_def_seq[valid_mask], y_prepay_seq[valid_mask]

X_test, masks_test = X_seq[test_mask], masks_seq[test_mask]
y_def_test, y_prepay_test = y_def_seq[test_mask], y_prepay_seq[test_mask]

print(f"Train: {X_train.shape[0]} loans")
print(f"Valid: {X_valid.shape[0]} loans")
print(f"Test: {X_test.shape[0]} loans")


Train: 31107 loans
Valid: 11376 loans
Test: 64629 loans
CPU times: user 91.1 ms, sys: 40.7 ms, total: 132 ms
Wall time: 135 ms


In [ ]:
#Fit LSTM
%%time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda')

class OptimizedLSTM(nn.Module):
    def __init__(self, input_size=5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 64, 2, batch_first=True, dropout=0.2)
        self.def_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
        self.prepay_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # [B, T, 64]
        def_haz = self.def_head(lstm_out).squeeze(-1)    # [B, T]
        prepay_haz = self.prepay_head(lstm_out).squeeze(-1)
        return def_haz, prepay_haz

class MaskedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce_def = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(25.0))
        self.bce_prepay = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(6.0))

    def forward(self, def_pred, prepay_pred, y_def, y_prepay, mask):
        # Mask unobserved months
        def_loss = self.bce_def(def_pred, y_def.float()) * mask.mean()
        prepay_loss = self.bce_prepay(prepay_pred, y_prepay.float()) * mask.mean()
        return def_loss + prepay_loss

# Data
train_ds = TensorDataset(
    torch.FloatTensor(X_train), torch.FloatTensor(y_def_train),
    torch.FloatTensor(y_prepay_train), torch.FloatTensor(masks_train)
)
train_loader = DataLoader(train_ds, 256, shuffle=True)

model = OptimizedLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
loss_fn = MaskedLoss()

print("Training optimized LSTM...")
for epoch in range(20):
    model.train()
    total_loss = 0
    for X_b, y_def_b, y_prepay_b, mask_b in train_loader:
        X_b, y_def_b, y_prepay_b, mask_b = [t.to(device) for t in (X_b, y_def_b, y_prepay_b, mask_b)]

        optimizer.zero_grad()
        def_pred, prepay_pred = model(X_b)
        loss = loss_fn(def_pred, prepay_pred, y_def_b, y_prepay_b, mask_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        print(f'Epoch {epoch}: {total_loss/len(train_loader):.4f}')


Training optimized LSTM...
Epoch 0: 0.6023
Epoch 5: 0.4573
Epoch 10: 0.4478
Epoch 15: 0.4467
CPU times: user 19 s, sys: 95.6 ms, total: 19 s
Wall time: 19.6 s


In [ ]:
%%time
# === LSTM TEST SET EVAL (2012+) ===
from sklearn.metrics import precision_recall_curve, auc
import numpy as np
import torch

def evaluate_lstm_test(model, X_test, y_def_test, y_prepay_test, masks_test):
    model.cpu()  # CPU safe
    model.eval()

    # Batch predictions (OOM safe)
    def_all, prepay_all = [], []
    for i in range(0, len(X_test), 32):
        X_b = torch.FloatTensor(X_test[i:i+32])
        with torch.no_grad():
            def_pred, prepay_pred = model(X_b)
            def_all.append(torch.sigmoid(def_pred).cpu().numpy())
            prepay_all.append(torch.sigmoid(prepay_pred).cpu().numpy())

    def_pred = np.concatenate(def_all)
    prepay_pred = np.concatenate(prepay_all)

    # Flatten TEST months only
    test_mask = masks_test > 0
    y_def_flat = y_def_test[test_mask].astype(bool)
    y_prepay_flat = y_prepay_test[test_mask].astype(bool)
    def_scores_flat = def_pred[test_mask]
    prepay_scores_flat = prepay_pred[test_mask]

    # PR-AUC
    def_prec, def_rec, _ = precision_recall_curve(y_def_flat, -def_scores_flat)
    prepay_prec, prepay_rec, _ = precision_recall_curve(y_prepay_flat, -prepay_scores_flat)

    def_pr_auc = auc(def_rec, def_prec)
    prepay_pr_auc = auc(prepay_rec, prepay_prec)

    # Lift curves
    def_top10 = lift_curve(y_def_flat, def_scores_flat)
    prepay_top10 = lift_curve(y_prepay_flat, prepay_scores_flat)

    print(f" LSTM TEST 2012 - 2014")
    print(f"Def PR-AUC: {def_pr_auc:.3f} | Top10: {def_top10:.1f}x")
    print(f"Prepay PR-AUC: {prepay_pr_auc:.3f} | Top10: {prepay_top10:.1f}x")

    return {'def_pr_auc': def_pr_auc, 'prepay_pr_auc': prepay_pr_auc,
            'def_top10': def_top10, 'prepay_top10': prepay_top10}

def lift_curve(y_true, y_scores, top_pct=0.1):
    top_threshold = np.percentile(y_scores, 100*(1-top_pct))
    top_mask = y_scores > top_threshold
    top_rate = y_true[top_mask].mean()
    pop_rate = y_true.mean()
    return top_rate / pop_rate if pop_rate > 0 else 1.0

# RUN TEST EVAL
test_results = evaluate_lstm_test(model, X_test, y_def_test, y_prepay_test, masks_test)


 LSTM TEST 2012 - 2014
Def PR-AUC: 0.011 | Top10: 1.4x
Prepay PR-AUC: 0.003 | Top10: 2.8x
CPU times: user 24 s, sys: 47.4 ms, total: 24 s
Wall time: 24.2 s


In [ ]:
# LSTM Test KS vs RF
from sklearn.metrics import roc_curve
import numpy as np
import torch

model.cpu(); model.eval()

# Test events (flatten masks)
test_mask_flat = masks_test.flatten() > 0
y_def_test_flat = y_def_test.flatten()[test_mask_flat].astype(int)
y_prepay_test_flat = y_prepay_test.flatten()[test_mask_flat].astype(int)

# LSTM predictions (flatten all)
lstm_def_all, lstm_prepay_all = [], []
for i in range(0, len(X_test), 16):
    X_b = torch.FloatTensor(X_test[i:i+16])
    with torch.no_grad():
        d_pred, p_pred = model(X_b)
        lstm_def_all.append(torch.sigmoid(d_pred).flatten().numpy())
        lstm_prepay_all.append(torch.sigmoid(p_pred).flatten().numpy())

lstm_def_scores = np.concatenate(lstm_def_all)[test_mask_flat]
lstm_prepay_scores = np.concatenate(lstm_prepay_all)[test_mask_flat]

def get_ks(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return max(np.abs(tpr - fpr))


#Note: RF scores already computed previously, but the displayed values are for an RF trained on 2005-2012 data. 
# The comparable model trained above on 2005-2010 has slightly worse values.
print("TEST 2012-2014 KS ")
print("Model     | Def KS | Prepay KS")
print("----------|--------|----------")
print(f"RF        | 0.057  | 0.670")
print(f"LSTM v2   | {get_ks(y_def_test_flat, lstm_def_scores):.3f} | {get_ks(y_prepay_test_flat, lstm_prepay_scores):.3f}")


TEST 2012-2014 KS 
Model     | Def KS | Prepay KS
----------|--------|----------
RF        | 0.057  | 0.670
LSTM v2   | 0.079 | 0.502


We conclude that RF and LSTM perform similarly, but LSTM does not dominate enough to justify its computational coomplexity.